In [1]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from numpy.linalg import norm
import datetime
import math
import multiprocess as mp
import time
import pickle
from pysb.simulator import BngSimulator
from pysb.simulator import ScipyOdeSimulator
from scipy.signal import find_peaks
import pyarrow.feather as feather
sys.path.insert(0, "../../Python/cellular_cycle")
from cell_cycle_death_model import model

In [2]:
PARAM = pd.read_csv(Path('../../Python/model_data/parameters.csv'))  # parameter file to use
INIT_CONCS = pd.read_csv(Path('../../Python/model_data/initial_concentrations.csv'))  # initial concentrations for simulations

TREATMENT = 'DMSO'  # set AKT, ERK, and FAK by setting 'ERL' or 'DMSO'

CYCDI_CONC = 0
POINTS = 10  # number to divide each condition into, sets the number of points to divide cell cycle
PROCESSES = mp.cpu_count() - 1  # number of pool processes to run at once
OVERWRITE = False  # Overwrite existing simulation files
VERBOSE = True
SET_JPARAM = False
PARAM_NUM = 0

integrator_options = {
    "rtol": 1e-12,
    "atol": 1e-12
}

# Where to save large simulation output files
SAVE_DIR = Path(Path.home(), 'ECM_mod_output')
if not SAVE_DIR.exists():
    SAVE_DIR.mkdir()

CURDATE = datetime.datetime.now().strftime('%Y%m%d')

In [3]:
def run_lyaps(AKT, ERK, FAK, np=np, model=model, points=POINTS, ScipyOdeSimulator=ScipyOdeSimulator,
              find_peaks=find_peaks, integrator_options=integrator_options, norm=norm, math=math,
              verbose=VERBOSE):
    model.parameters['AKT'].value = AKT
    model.parameters['ERK'].value = ERK
    model.parameters['FAK'].value = FAK
    ep = 0.001 * 200
    t = np.linspace(0, 1000, 1000001)
    t_0 = np.linspace(0, 2000, 2000001)
    all_logz = dict.fromkeys(list(range(1, points + 1)))

    for kk in range(points):
        sim_ode = ScipyOdeSimulator(model, t_0, verbose=False, compiler='python',
                                    integrator_options=integrator_options).run()
        y_out = sim_ode.observables

        peaks_cdc, _ = find_peaks(y_out['obsCdc20'], prominence=0.6, distance=10)
        # new_y = y_out[peaks_cdc[-11]:peaks_cdc[-2]]
        peaks_half = peaks_cdc[peaks_cdc < len(y_out)/2]
        peaks_half2 = peaks_cdc[peaks_cdc > len(y_out) / 2]

        if len(peaks_half2) > 3:
            IMTs = [t_0[y] - t_0[x] for x, y in zip(peaks_half2, peaks_half2[1:])]
            meanIMT = sum(IMTs) / len(IMTs)
            variable_range = y_out[peaks_half[-3]:peaks_half[-2]]
            tmp_loc = (round(len(variable_range) / (points)) * (kk))
        else:
            meanIMT = 0
            variable_range = y_out[575000:600000]
            tmp_loc = (round(len(variable_range) / (points)) * (kk))
        if verbose:
            print(meanIMT)


        Initial_1 = variable_range[tmp_loc]
        Initial_1 = [val + ep for val in Initial_1]

        prots = ['Skp2_0', 'p21_0', 'p21p_0', 'cdh1act_0', 'cdh1inact_0', 'Myc_0', 'CycD_0', 'CycDp21_0', 'pRb_0',
                 'pRbp_0', 'E2F_0', 'pRbE2F_0', 'CycE_0', 'CycEp21_0', 'CycA_0', 'CycAp21_0', 'CycB_0', 'CycBp21_0',
                 'Cdc20_0']
        prot_init = {prots[i]: Initial_1[i] for i in range(len(prots))}

        for key in prot_init.keys():
            model.parameters[key].value = prot_init[key] / model.parameters['omega'].value

        sim_ode1 = ScipyOdeSimulator(model, t, verbose=False, compiler='python',
                                     integrator_options=integrator_options).run()
        y_out1 = sim_ode1.observables

        z = [0] * (round((len(y_out1) / 2000)) + 1)
        n = 0
        for i in range(0, len(y_out1), 2000):
            if len(peaks_half2) > 0:
                window = y_out[peaks_half[-3]+i+tmp_loc-5000:peaks_half[-2]+i+tmp_loc+5000]
            else:
                window = y_out[575000+i+tmp_loc-5000:600000+i+tmp_loc+5000]

            dists = [0] * len(window)
            for j in range(0, len(window)):
                dif = np.asarray(window[j].tolist())-np.asarray(y_out1[i].tolist())
                dists[j] = norm(dif[0:19]/200)

            z[n] = min(dists)
            n = n + 1

        logz = [math.log(num) for num in z]
        all_logz[kk + 1] = logz
        if verbose:
            print([AKT, ERK, FAK, kk])

    return all_logz, meanIMT

In [4]:
t = np.linspace(0, 1000, 1000001)


In [5]:
model.parameters['AKT'].value = 1
model.parameters['ERK'].value = 1
model.parameters['FAK'].value = 1

# Importing the parameters:

for key in PARAM.keys():
    try:
        if model.parameters[key].value != PARAM[key][0]:
            model.parameters[key].value = PARAM[key]
    except:
        print(key)
model.parameters['omega'].value = 200


if SET_JPARAM:
    with open('fit_data/pso/frac_prolif_parameters_pso.pkl', 'rb') as f:
        frac_prolif_parameters_pso = pickle.load(f)
    new_param_pso_sort = frac_prolif_parameters_pso.sort_values([('cost', 'total cost likelihood')])
    new_params = new_param_pso_sort['model parameters']
    # Importing the parameters:
    for key in new_params.keys():
        try:
            if model.parameters[key].value != new_params[key].iloc[PARAM_NUM]:
                model.parameters[key].value = new_params[key].iloc[PARAM_NUM]
        except:
            print(key)
    model.parameters['omega'].value = 200

for key in INIT_CONCS.keys():
    try:
        model.parameters[key].value = INIT_CONCS[key][0] / model.parameters['omega'].value
    except:
        print(key)

model.parameters['CycDi_0'].value = CYCDI_CONC

if TREATMENT == 'ERL':
    FAKs = list(np.linspace(0.4,0.6,50))
    tmp = list(np.linspace(0.6,1,50))
    FAKs.extend(tmp)
    AKTs = list(np.linspace(0.3,0.6,50))
    tmp = [0.6]*50
    AKTs.extend(tmp)
    ERKs = [0.2]*100
else:
    FAKs = list(np.linspace(0.4,0.6,50))
    tmp = list(np.linspace(0.6,1,50))
    FAKs.extend(tmp)
    AKTs = list(np.linspace(0.6,1,50))
    tmp = [1]*50
    AKTs.extend(tmp)
    ERKs = [1]*100

conds = [(AKT, ERK, FAK) for AKT, ERK, FAK, in zip(AKTs, ERKs, FAKs)]

Stress
b
Hippo_tot
Yap_tot
AKT_tot
ERK_tot
FAK_tot
skp2_tot
p21_tot
cdh1_tot
CycD_tot
K_1E2F
V_1E2F
k_cd4
k_myc1
k_CE
d_CE
K_i13
K_i14
V_sac
V_da
V_sec
V_de
V_sf
V_fs
V_dfa
v_scdh1a
v_2fak
v_s2p21
K_AFAK
K_dfa
K_fs
K_cf
K_cac
K_cec
K_da
K_de
K_sp21
k_dcdh1a
K_p21fak.1
k_CDF
K_DF
K_dcescp2
BAXm_0
BCL2_0


In [6]:
start = time.time()
with mp.Pool(processes = PROCESSES) as p:
    results = p.starmap(run_lyaps, conds)
end = time.time()
print(f'Process took {np.round(end - start, 2)} seconds')

24.010219512195125
24.091121951219513
25.900315789473687
24.134400000000003
25.47721052631579
24.39724.1094

26.359432432432435
24.728102564102564
23.917525
25.087307692307693
[0.8448979591836735, 1, 0.5224489795918368, 0]
[0.7224489795918367, 1, 0.4612244897959184, 0]
[0.6979591836734693, 1, 0.4489795918367347, 0]
[0.746938775510204, 1, 0.47346938775510206, 0]
[0.673469387755102, 1, 0.43673469387755104, 0]
[0.6489795918367347, 1, 0.4244897959183674, 0]
[0.7714285714285714, 1, 0.4857142857142857, 0]
[0.6244897959183673, 1, 0.4122448979591837, 0]
[0.6, 1, 0.4, 0]
[0.7959183673469388, 1, 0.49795918367346936, 0]
[0.8204081632653062, 1, 0.5102040816326531, 0]
23.848390243902443
24.396974999999998
24.727794871794874
24.093292682926826
25.087051282051284
24.1336
25.47697368421053
25.90013157894737
24.147100000000002
26.359324324324326
24.075025000000004
[0.8448979591836735, 1, 0.5224489795918368, 1]
[0.8204081632653062, 1, 0.5102040816326531, 1]
[0.7959183673469388, 1, 0.49795918367346936, 1

In [7]:
z_list = [num[0] for num in results]
mean_IMT = [num[1] for num in results]
mean_IMT

[26.35843243243243,
 26.199815789473686,
 26.04518918918919,
 25.898736842105265,
 25.751973684210526,
 25.608947368421056,
 25.475236842105264,
 25.339473684210525,
 25.20723076923077,
 25.085076923076926,
 24.95953846153846,
 24.837358974358974,
 24.72569230769231,
 24.60975,
 24.496975,
 24.394775000000003,
 24.28792682926829,
 24.183225,
 24.0933,
 24.060025,
 24.095525000000002,
 24.12885,
 24.13655,
 24.136300000000002,
 24.142999999999997,
 24.124474999999997,
 24.09705,
 24.072074999999998,
 23.95909756097561,
 23.90317073170732,
 23.84656097560976,
 23.912975609756096,
 23.69587804878049,
 23.614878048780483,
 23.52360975609756,
 23.427390243902437,
 23.394380952380953,
 23.288357142857144,
 23.17930952380953,
 23.068261904761904,
 22.955285714285715,
 22.789953488372095,
 22.6793488372093,
 22.570790697674415,
 22.465069767441857,
 22.399477272727275,
 22.29840909090909,
 22.203636363636363,
 22.115909090909092,
 22.048295454545457,
 22.048295454545457,
 22.02034090909091,
 2

In [8]:
z_list = [num[0] for num in results]
mean_IMT = [num[1] for num in results]

df = pd.DataFrame(z_list)
mean_df = pd.DataFrame(mean_IMT)

cycdi = str(round(model.parameters['CycDi_0'].value % 1,2)).split('.')[1]

In [9]:
if SET_JPARAM:
    filename = str(Path(SAVE_DIR, f'lyaps_{TREATMENT}_pso_{PARAM_NUM}_cycdi_0_{cycdi}.fea')) # feather might have an issue with Path object
else:
    filename = str(Path(SAVE_DIR, f'lyaps_{TREATMENT}_cycdi_0_{cycdi}.fea'))
feather.write_feather(df, filename)

if SET_JPARAM:
    filename = str(Path(SAVE_DIR, f'mINT_{TREATMENT}_pso_{PARAM_NUM}_cycdi_0_{cycdi}.fea'))
else:
    filename = str(Path(SAVE_DIR, f'mINT_{TREATMENT}_cycdi_0_{cycdi}.fea'))

feather.write_feather(mean_df, filename)